# Use Case — Prioritizing Bus-Stop Cooling Interventions

**Who this is for**  
Urban planners, transit-authority analysts, and climate-adaptation leads who have *their own* infrastructure data (bus stops, public benches, playgrounds, bike-share docks, shelters, schools) and need to decide **which locations to treat first** when budget is limited.

**The scenario**  
Summer heat is making your city's bus stops unbearable. Ridership dips, complaints pile up, and the council has just approved a cooling-intervention budget — trees, shade structures, reflective pavement. You have a list of bus stops. You do **not** have the budget to treat all of them. You need a data-backed, defensible shortlist.

This notebook combines **your data** (a bus-stops point layer) with **FortyGuard's layers** (heatmap, satellite segmentation, street view, environmental parameters) to answer four questions in sequence:

1. **Which stops are actually hot?**  ← heatmap × your points
2. **Why are they hot?**  ← satellite segmentation on the top hotspots
3. **What does that look like on the ground?**  ← street view on the #1 stop
4. **When is heat at its worst here?**  ← environmental parameters profile

The final output is a prioritized action list — one row per stop, ranked by temperature, with a dominant cause and a recommended intervention.

> **Bring your own data.** The notebook ships with a sample CSV at `data/sample_bus_stops.csv`. Swap in your own CSV with the same columns (`stop_id`, `name`, `latitude`, `longitude`) at Step 1 and everything downstream just works.

---

## Setup

Load `.env`, instantiate the client, define the study area. Run `notebooks/00_setup.ipynb` first if this cell errors out.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parents[1]
sys.path.insert(0, str(ROOT))

from dotenv import load_dotenv
load_dotenv(ROOT / '.env')

import pandas as pd
import folium
import matplotlib.pyplot as plt
from shapely.geometry import Point, shape

from fortyguard import FortyGuardClient
from fortyguard.samples import SAN_JOSE_POLYGON

client = FortyGuardClient()

# Study parameters. Change the date/time for the hour you want to prioritize on.
AOI              = SAN_JOSE_POLYGON      # ~104 km² (~40 mi²) across central San Jose
STUDY_DATE       = '2024-07-15'
STUDY_HOUR       = '14:00'               # design-peak afternoon
GRANULARITY_M    = 100                   # 100 m → ~10 k tiles over this AOI; 80 m would be ~16 k
TOP_N_TO_DIAGNOSE = 3                    # deepen analysis on this many hottest stops

print(f'Authenticated to {client.base_url}')
print(f'Study hour: {STUDY_DATE} {STUDY_HOUR}')

---
## Step 1 — Load your data

### What you are doing
Reading a bus-stops point layer from CSV. The schema is minimal — `stop_id`, `name`, `latitude`, `longitude` — so you can export this directly from the transit agency's GIS, a GTFS feed, or a spreadsheet.

### Why this matters
Everything downstream is built around the geometry of **your** assets. By starting from your own data, the outputs land in your existing workflow: same IDs, same names, same coordinate system. That is the difference between a dashboard and something the operations team can act on.

In [ ]:
# Swap this path for your own CSV — same four columns.
stops = pd.read_csv(ROOT / 'data' / 'sample_bus_stops.csv')
print(f'Loaded {len(stops)} stops')
stops.head()

In [ ]:
# Plot the stops as a quick sanity check.
center = [stops['latitude'].mean(), stops['longitude'].mean()]
fmap = folium.Map(location=center, zoom_start=12, tiles='cartodbpositron')
for _, r in stops.iterrows():
    folium.CircleMarker(
        location=[r.latitude, r.longitude],
        radius=5, color='#1f77b4', fill=True, fill_opacity=0.9,
        popup=f"{r.stop_id} — {r['name']}",
    ).add_to(fmap)
folium.GeoJson(AOI, style_function=lambda _: {'color': '#555', 'fill': False, 'weight': 1, 'dashArray': '5,5'}).add_to(fmap)
fmap.fit_bounds([[stops['latitude'].min(), stops['longitude'].min()],
                 [stops['latitude'].max(), stops['longitude'].max()]])
fmap

---
## Step 2a — Create the heat layer (via API)

### What you are doing
Requesting a high-resolution heatmap over the study AOI at the design-peak hour. The response is a GeoJSON tile layer with a temperature value on every tile.

### Why this matters
Weather stations give you one number for the whole city. A heatmap gives you temperature **at the spatial resolution your decisions are made** — block by block. That is what lets you separate hot stops from merely-average stops.

> Run **either** Step 2a (live API call, consumes credits) **or** Step 2b (load a cached sample, free). Both produce the same `map_data` / `features` / `t_stats` variables, so everything downstream works identically.

In [ ]:
heatmap = client.create_heatmap(
    polygon_aoi=AOI,
    start_date=STUDY_DATE,
    start_time=STUDY_HOUR,
    filter_type=1,
    granularity=GRANULARITY_M,
)

map_data = heatmap['result'].get('map_data') or {}
features = map_data.get('features', []) if isinstance(map_data, dict) else []

stats = heatmap['result'].get('stats_data', {})
t_stats = stats.get('Temperature_stats') or stats.get('temperature_stats') or {}
print(f'Tiles returned       : {len(features)}')
print(f'AOI temperature stats: {t_stats}')

---
## Step 2b — Or: load a pre-generated heatmap (for testing)

### What you are doing
Loading a cached heatmap from `data/san_jose_heatmap_sample.geojson` instead of calling the API. Tile properties (`min_temperature`, `max_temperature`, `average_temperature` in °F) are normalized into a single `temperature` property in °C so the rest of the notebook sees the same shape Step 2a would produce.

### Why this matters
Use this path when iterating on the downstream logic without burning API credits on a heatmap you already have. Skip it on a real run — Step 2a gives you a heatmap for the exact date, hour, and AOI you care about.

In [ ]:
import json

HEATMAP_PATH = ROOT / 'data' / 'san_jose_heatmap_sample.geojson'
with open(HEATMAP_PATH, 'r') as f:
    map_data = json.load(f)

# Normalize each feature so it carries a single `temperature` (°C) property,
# mirroring what client.create_heatmap(...) would return.
def _f_to_c(t):
    return None if t is None else round((float(t) - 32) * 5 / 9, 2)

features = map_data.get('features', [])
for feat in features:
    props = feat.setdefault('properties', {})
    avg_f = props.get('average_temperature')
    props['temperature'] = _f_to_c(avg_f)

temps = [f['properties']['temperature'] for f in features if f['properties'].get('temperature') is not None]
t_stats = {
    'count': len(temps),
    'min': round(min(temps), 2) if temps else None,
    'max': round(max(temps), 2) if temps else None,
    'mean': round(sum(temps) / len(temps), 2) if temps else None,
}
print(f'Loaded heatmap       : {HEATMAP_PATH.name}')
print(f'Tiles returned       : {len(features)}')
print(f'AOI temperature stats: {t_stats}  (°C)')

---
## Step 3 — Correlate your data with the heat layer

### What you are doing
For each bus stop, finding the heatmap tile that contains it and copying that tile's temperature onto the stop. This is the **spatial join** — the moment where your asset layer and our thermal layer merge into one table.

### Why this matters
Before this step, "it's hot in the city" was a general observation. After this step, every row in your bus-stops table carries a specific temperature at the design hour. You can now sort, filter, group by route, report by neighborhood — anything you would normally do with your operational data.

In [ ]:
# Build shapely geometries once, then assign each stop its containing tile temperature.
tile_polys = [(shape(f['geometry']), f['properties'].get('temperature')) for f in features]

def _stop_temperature(lat: float, lon: float):
    p = Point(lon, lat)
    # Preferred: tile that contains the point.
    for poly, temp in tile_polys:
        if poly.contains(p):
            return temp
    # Fallback: nearest tile by centroid distance.
    if not tile_polys:
        return None
    nearest = min(tile_polys, key=lambda pt: pt[0].centroid.distance(p))
    return nearest[1]

stops['temperature_c'] = stops.apply(
    lambda r: _stop_temperature(r['latitude'], r['longitude']), axis=1
)
stops[['stop_id', 'name', 'temperature_c']].head()

---
## Step 4 — Rank and visualize

### What you are doing
Sorting stops by the temperature value we just attached, then plotting them on a map with the heatmap tiles as the backdrop. Marker color scales with stop temperature; hottest stops jump out visually and land at the top of the table.

### Why this matters
This is the first concrete deliverable — a ranked short-list of candidate locations. Even without the downstream diagnostic steps, this alone is more actionable than any citywide average the council has seen.

In [ ]:
ranked = stops.sort_values('temperature_c', ascending=False).reset_index(drop=True)
ranked.insert(0, 'rank', ranked.index + 1)
ranked

In [ ]:
# Map: heatmap tiles in the background, bus stops colored by temperature on top.
lo, hi = ranked['temperature_c'].min(), ranked['temperature_c'].max()

def _tile_style(feat):
    t = feat['properties'].get('temperature', lo)
    frac = 0 if hi == lo else (t - lo) / (hi - lo)
    r, b = int(255*frac), int(255*(1-frac))
    return {'fillColor': f'#{r:02x}00{b:02x}', 'color': '#00000000', 'fillOpacity': 0.35, 'weight': 0}

def _stop_color(t):
    if t is None: return '#888'
    frac = 0 if hi == lo else (t - lo) / (hi - lo)
    r, b = int(255*frac), int(255*(1-frac))
    return f'#{r:02x}00{b:02x}'

fmap = folium.Map(location=center, zoom_start=12, tiles='cartodbpositron')
if features:
    folium.GeoJson(map_data, style_function=_tile_style).add_to(fmap)
for _, r in ranked.iterrows():
    folium.CircleMarker(
        location=[r.latitude, r.longitude],
        radius=8, color='black', weight=1,
        fill=True, fill_color=_stop_color(r.temperature_c), fill_opacity=0.95,
        popup=f"#{r['rank']}  {r['stop_id']} — {r['name']}<br/>{r.temperature_c:.1f} °C",
    ).add_to(fmap)
fmap.fit_bounds([[ranked['latitude'].min(), ranked['longitude'].min()],
                 [ranked['latitude'].max(), ranked['longitude'].max()]])
fmap

---
## Step 5 — Zoom in on above-average hotspots

### What you are doing
Filtering both the heatmap tiles and the bus stops to the band `mean < temperature ≤ max` and redrawing the same map. The cooler half of the AOI falls away; only the genuinely-hot tiles and the stops inside them remain.

### Why this matters
When every tile is drawn, the eye gets pulled to whatever is warmest in the visible frame — which may still be near the citywide average. Filtering to above-mean sharpens the question: *of the stops that are hotter than typical for the AOI at this hour, where are they clustered?* That's the cluster map the council should see first.

In [ ]:
mean_t = t_stats['mean']
max_t  = t_stats['max']

hot_features = [
    f for f in features
    if f['properties'].get('temperature') is not None
    and f['properties']['temperature'] > mean_t
    and f['properties']['temperature'] <= max_t
]

hot_stops = ranked[(ranked['temperature_c'] > mean_t) &
                   (ranked['temperature_c'] <= max_t)].copy()

# Keep only the tiles that intersect at least one hot stop.
# Use a tiny buffer around each point so boundary points are not missed.
stop_points = [Point(r.longitude, r.latitude).buffer(1e-6) for _, r in hot_stops.iterrows()]
hot_features = [
    f for f in hot_features
    if any(shape(f['geometry']).intersects(p) for p in stop_points)
]
hot_map_data = {'type': 'FeatureCollection', 'features': hot_features}

print(f'AOI mean: {mean_t:.2f} °C   max: {max_t:.2f} °C')
print(f'Tiles intersecting hot stops: {len(hot_features)}')
print(f'Stops above mean: {len(hot_stops)} / {len(ranked)}')

# Print centroid coordinates of each hot tile.
for i, f in enumerate(hot_features):
    centroid = shape(f['geometry']).centroid
    print(f'  Tile {i+1}: centroid ({centroid.y:.6f}, {centroid.x:.6f})')

fmap_hot = folium.Map(location=center, zoom_start=12, tiles='cartodbpositron')
if hot_features:
    folium.GeoJson(hot_map_data, style_function=_tile_style).add_to(fmap_hot)
for _, r in hot_stops.iterrows():
    folium.CircleMarker(
        location=[r.latitude, r.longitude],
        radius=8, color='black', weight=1,
        fill=True, fill_color=_stop_color(r.temperature_c), fill_opacity=0.95,
        popup=f"#{r['rank']}  {r['stop_id']} — {r['name']}<br/>{r.temperature_c:.1f} °C",
    ).add_to(fmap_hot)
if len(hot_stops):
    fmap_hot.fit_bounds([[hot_stops['latitude'].min(), hot_stops['longitude'].min()],
                         [hot_stops['latitude'].max(), hot_stops['longitude'].max()]])
fmap_hot

---
## Step 6 — Diagnose the top hotspots (why are they hot?)

### What you are doing
Running satellite segmentation on the top-N hottest stops. The API classifies the surroundings of each point into surface classes (rooftops, roads, vegetation, water, bare land). We collect the percentages into the same DataFrame.

### Why this matters
Knowing a stop is hot is not actionable on its own — *intervention selection depends on the cause*. Planting trees fixes a low-vegetation problem; reflective paving fixes a high-impervious problem; a shade structure fixes a high sky-exposure problem. Satellite segmentation tells you which of those is the dominant driver for each candidate stop.

In [ ]:
IMPERVIOUS_KEYS = {'road', 'roads', 'pavement', 'building', 'buildings', 'rooftop', 'rooftops', 'bare'}
VEGETATION_KEYS = {'vegetation', 'tree', 'trees', 'grass', 'greenery', 'park'}

def _bucket(segments: dict, keys: set) -> float:
    total = 0.0
    for cls, pct in segments.items():
        if any(k in cls.lower() for k in keys):
            try:
                total += float(pct)
            except (TypeError, ValueError):
                pass
    return round(total, 1)

top = ranked.head(TOP_N_TO_DIAGNOSE).copy()
impervious, vegetation, raw_segments = [], [], []

for _, r in top.iterrows():
    print(f"Diagnosing #{r['rank']}  {r['stop_id']} — {r['name']}")
    sat = client.satellite_segmentation(
        latitude=r.latitude, longitude=r.longitude,
        start_date=STUDY_DATE, start_time=STUDY_HOUR,
        filter_type=1, granularity=GRANULARITY_M,
        verbose=False,
    )
    segs = sat['result'].get('segmentation', {}).get('segments', {}) or {}
    raw_segments.append(segs)
    impervious.append(_bucket(segs, IMPERVIOUS_KEYS))
    vegetation.append(_bucket(segs, VEGETATION_KEYS))

top['impervious_pct'] = impervious
top['vegetation_pct'] = vegetation
top[['rank', 'stop_id', 'name', 'temperature_c', 'impervious_pct', 'vegetation_pct']]

---
## Step 6b — Load cached satellite segmentation for a hot tile

### What you are doing
Reading a pre-saved satellite segmentation result from `data/satellite_segmentation_urban_planner.json`. This file was produced by running satellite segmentation at the centroid of one of the hot tiles identified in Step 5. We display the coordinates, the original satellite image, the segmented image, the segmentation percentages, and the location on a map.

### Why this matters
Use this path when you already have a cached segmentation result and want to inspect it without burning API credits. The file contains the original image, the segmented image, and the class breakdown — everything you need to understand why a tile is hot.

In [ ]:
import json, base64, io
from PIL import Image

SAT_SEG_PATH = ROOT / 'data' / 'satellite_segmentation_urban_planner.json'
with open(SAT_SEG_PATH, 'r') as f:
    sat_data = json.load(f)

# --- Coordinates ---
coords = sat_data['coordinates']
lat, lon = float(coords['latitude']), float(coords['longitude'])
print(f"Location: ({lat}, {lon})")
print(f"Image year: {sat_data.get('image_year', 'N/A')}")

# --- Segmentation results ---
seg = sat_data['segmentation']
segments = seg['segments']
print(f"\nSegmentation results:")
for cls, pct in segments.items():
    print(f"  {cls:>12s}: {pct}%")

# --- Decode images ---
def _decode_b64(b64_str):
    if not b64_str:
        return None
    if isinstance(b64_str, list):
        b64_str = b64_str[0]
    if b64_str.startswith('data:'):
        b64_str = b64_str.split(',', 1)[1]
    return Image.open(io.BytesIO(base64.b64decode(b64_str)))

original_img = _decode_b64(sat_data.get('orignal_image'))
segmented_img = _decode_b64(seg.get('image_content'))

# --- Display images side by side ---
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
if original_img is not None:
    axes[0].imshow(original_img)
axes[0].set_title(f"Original satellite image ({lat:.4f}, {lon:.4f})")
axes[0].axis('off')

if segmented_img is not None:
    axes[1].imshow(segmented_img)
axes[1].set_title("Segmented image")
axes[1].axis('off')
plt.tight_layout()
plt.show()

# --- Legend ---
legend = seg.get('image_legend', {})
if legend:
    print("\nLegend (RGB):")
    for cls, rgb in legend.items():
        pct = segments.get(cls, '?')
        print(f"  {cls:>12s}: rgb{tuple(rgb)}  — {pct}%")

# --- Show location on map ---
fmap_seg = folium.Map(location=[lat, lon], zoom_start=16, tiles='cartodbpositron')
folium.Marker(
    location=[lat, lon],
    popup=f"Satellite segmentation<br>({lat:.6f}, {lon:.6f})",
    icon=folium.Icon(color='red', icon='info-sign'),
).add_to(fmap_seg)
folium.Circle(
    location=[lat, lon], radius=GRANULARITY_M / 2,
    color='red', fill=True, fill_opacity=0.15,
    popup=f"~{GRANULARITY_M}m tile",
).add_to(fmap_seg)

# --- Populate top / impervious_pct / vegetation_pct for Step 9 (cached path). ---
# The cached segmentation covers one tile, so we apply its percentages to the
# top-N hottest stops as a demonstration — on the live path Step 6 runs
# segmentation per-stop and each row gets its own numbers.
IMPERVIOUS_KEYS = {'road', 'roads', 'pavement', 'building', 'buildings', 'rooftop', 'rooftops', 'bare'}
VEGETATION_KEYS = {'vegetation', 'tree', 'trees', 'grass', 'greenery', 'park'}

def _bucket(segs, keys):
    total = 0.0
    for cls, pct in segs.items():
        if any(k in cls.lower() for k in keys):
            try: total += float(pct)
            except (TypeError, ValueError): pass
    return round(total, 1)

top = ranked.head(TOP_N_TO_DIAGNOSE).copy()
top['impervious_pct'] = _bucket(segments, IMPERVIOUS_KEYS)
top['vegetation_pct'] = _bucket(segments, VEGETATION_KEYS)
print(f"\nPopulated top (n={len(top)}) for Step 9 — "
      f"impervious {top['impervious_pct'].iloc[0]}%, vegetation {top['vegetation_pct'].iloc[0]}%")

fmap_seg

---
## Step 7 — Ground-truth the #1 stop with street view

### What you are doing
Running street view segmentation at the hottest stop, oriented down the street. The API returns a ground-level image plus a pixel-wise segmentation.

### Why this matters
Satellite view shows *surroundings from above*. Street view shows *what a rider waiting at that stop actually sees*. That perspective is where you confirm whether a shade structure is feasible, whether there is room for trees, and whether the shelter itself (a metal box that absorbs heat) is part of the problem.

In [ ]:
hot1 = top.iloc[0]
street = client.street_view_segmentation(
    latitude=hot1.latitude, longitude=hot1.longitude,
    vertical_angle=5.0, horizontal_angle=0.0, back_view=False,
    verbose=False,
)
front = street['result'].get('front', {})
front_segments = front.get('segments', {}) or {}

sky_pct = _bucket(front_segments, {'sky'})
building_pct = _bucket(front_segments, {'building', 'buildings', 'wall'})
print(f"#1 stop: {hot1['stop_id']} — {hot1['name']}")
print(f"  sky fraction     : {sky_pct}%  (→ higher = more shade needed)")
print(f"  building fraction: {building_pct}%  (→ higher = more self-shading already)")

In [ ]:
import base64, io
from PIL import Image

def _decode(b64):
    if not b64: return None
    if b64.startswith('data:'): b64 = b64.split(',', 1)[1]
    return Image.open(io.BytesIO(base64.b64decode(b64)))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, key, title in [(axes[0], 'original_image',  f"Street view at {hot1['stop_id']}"),
                       (axes[1], 'segmented_image', 'Segmentation')]:
    img = _decode(front.get(key))
    if img is not None: ax.imshow(img)
    ax.set_title(title); ax.axis('off')
plt.tight_layout(); plt.show()

---
## Step 7b — Load cached street view segmentation for a hot tile

### What you are doing
Reading a pre-saved street view segmentation result from `data/street_view_segmentation_urban_planner.json`. This file was produced by running street view segmentation at the centroid of one of the hot tiles identified earlier. We display the coordinates, the original street view image, the segmented image, the segmentation percentages, and the location on a map.

### Why this matters
Use this path when you already have a cached street view result and want to inspect it without burning API credits. The file contains the ground-level original image, the pixel-wise segmented image, and the class breakdown — everything you need to confirm on-the-ground conditions at a hotspot.

In [ ]:
import json, base64, io
from PIL import Image

STREET_SEG_PATH = ROOT / 'data' / 'street_view_segmentation_urban_planner.json'
with open(STREET_SEG_PATH, 'r') as f:
    street_data = json.load(f)

# --- Coordinates ---
coords = street_data['coordinates']
lat, lon = float(coords['latitude']), float(coords['longitude'])
print(f"Location: ({lat}, {lon})")
print(f"Image date: {street_data.get('front', {}).get('image_date', 'N/A')}")

# --- Segmentation results ---
front = street_data['front']
segments = front['segments']
print(f"\nSegmentation results:")
for cls, pct in segments.items():
    print(f"  {cls:>12s}: {pct}%")

# --- Decode images ---
def _decode_b64(b64_str):
    if not b64_str:
        return None
    if isinstance(b64_str, list):
        b64_str = b64_str[0]
    if b64_str.startswith('data:'):
        b64_str = b64_str.split(',', 1)[1]
    return Image.open(io.BytesIO(base64.b64decode(b64_str)))

original_img = _decode_b64(front.get('original_image'))
segmented_img = _decode_b64(front.get('segmented_image'))

# --- Display images side by side ---
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
if original_img is not None:
    axes[0].imshow(original_img)
axes[0].set_title(f"Original street view ({lat:.4f}, {lon:.4f})")
axes[0].axis('off')

if segmented_img is not None:
    axes[1].imshow(segmented_img)
axes[1].set_title("Segmented image")
axes[1].axis('off')
plt.tight_layout()
plt.show()

# --- Legend ---
legend = front.get('image_legend', {})
if legend:
    print("\nLegend (RGB):")
    for cls, rgb in legend.items():
        pct = segments.get(cls, '?')
        print(f"  {cls:>12s}: rgb{tuple(rgb)}  — {pct}%")

# --- Show location on map ---
fmap_street = folium.Map(location=[lat, lon], zoom_start=17, tiles='cartodbpositron')
folium.Marker(
    location=[lat, lon],
    popup=f"Street view segmentation<br>({lat:.6f}, {lon:.6f})",
    icon=folium.Icon(color='red', icon='info-sign'),
).add_to(fmap_street)
folium.Circle(
    location=[lat, lon], radius=GRANULARITY_M / 2,
    color='red', fill=True, fill_opacity=0.15,
    popup=f"~{GRANULARITY_M}m tile",
).add_to(fmap_street)

# --- Populate sky_pct for Step 9 (cached-path compatibility). ---
def _bucket(segs, keys):
    total = 0.0
    for cls, pct in segs.items():
        if any(k in cls.lower() for k in keys):
            try: total += float(pct)
            except (TypeError, ValueError): pass
    return round(total, 1)

sky_pct = _bucket(segments, {'sky'})
print(f"\nsky_pct (for Step 9): {sky_pct}%")

fmap_street

---
## Step 8 — Environmental drivers through the day

### What you are doing
Profiling environmental parameters at the #1 stop from 07:00 to 19:00 on the study date. We plot heat index and relative humidity through the day so you can see *when* discomfort peaks, not just how hot it gets at one instant.

### Why this matters
A stop that is unbearable from 12:00–17:00 demands different intervention timing than one that peaks during evening commute. The hour-by-hour profile tells you whether shade, misting, or ventilation is the right fix — and whether the fix needs to be passive (works all day) or active (runs only during peak).

In [ ]:
env = client.environmental_parameters(
    latitude=hot1.latitude, longitude=hot1.longitude,
    temperature=float(hot1.temperature_c),
    start_date=STUDY_DATE, start_time='07:00', end_time='19:00',
    filter_type=2, verbose=False,
)
res    = env['result']
loc    = res['locations'][0]
params = loc.get('parameters', {})
ts     = pd.to_datetime(res['metadata'].get('timestamps', []))

env_df = pd.DataFrame({k: v for k, v in params.items()
                       if isinstance(v, list) and len(v) == len(ts)})
env_df.insert(0, 'timestamp', ts); env_df.set_index('timestamp', inplace=True)

plot_cols = [c for c in ('heat_index_celsius', 'apparent_temperature_celsius',
                         'wet_bulb_temperature_celsius', 'relative_humidity_percent')
             if c in env_df.columns]
if plot_cols:
    fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
    thermal = [c for c in plot_cols if 'humidity' not in c]
    if thermal: env_df[thermal].plot(ax=axes[0], marker='o'); axes[0].set_title(f"Thermal comfort at {hot1['stop_id']}"); axes[0].set_ylabel('°C'); axes[0].grid(alpha=0.3)
    if 'relative_humidity_percent' in env_df.columns:
        env_df['relative_humidity_percent'].plot(ax=axes[1], color='steelblue', marker='o')
        axes[1].set_title('Relative humidity'); axes[1].set_ylabel('%'); axes[1].grid(alpha=0.3)
    plt.tight_layout(); plt.show()

peak_hi = env_df['heat_index_celsius'].max() if 'heat_index_celsius' in env_df.columns else None
peak_hour = env_df['heat_index_celsius'].idxmax() if 'heat_index_celsius' in env_df.columns else None
print(f"Peak heat index: {peak_hi}  at {peak_hour}")

---
## Step 8b — Load cached environmental parameters for a hot tile

### What you are doing
Reading a pre-saved environmental parameters result from `data/env_parameters_urban_planner.json`. This file was produced by running environmental parameters at the centroid of one of the hot tiles identified earlier. We display the coordinates, elevation, the time range covered, the parameter values (heat index, apparent temperature, wet-bulb, humidity, air quality, etc.), the solar irradiance, and the location on a map.

### Why this matters
Use this path when you already have a cached environmental parameters snapshot and want to inspect it without burning API credits. The file contains the full parameter set at the requested time(s) — everything you need to understand the environmental drivers at a hotspot.

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import folium

ENV_PATH = ROOT / 'data' / 'env_parameters_urban_planner.json'
with open(ENV_PATH, 'r') as f:
    env_data = json.load(f)

# --- Location ---
loc_cached = env_data['locations'][0]
lat, lon = float(loc_cached['lat']), float(loc_cached['lon'])
elevation = loc_cached.get('elevation')
tile_temp = loc_cached.get('temperature')
print(f"Location    : ({lat}, {lon})")
print(f"Elevation   : {elevation} m")
print(f"Tile temp   : {tile_temp} °C")

# --- Time range ---
meta = env_data.get('metadata', {})
tr = meta.get('time_range', {})
print(f"Time range  : {tr.get('start')} → {tr.get('end')}  (interval: {tr.get('interval')}, count: {tr.get('count')})")

ts = pd.to_datetime(meta.get('timestamps', []))
params = loc_cached.get('parameters', {})
solar_cs = loc_cached.get('solar_irradiance', {}).get('clear_sky', {})

env_df = pd.DataFrame({k: v for k, v in params.items()
                       if isinstance(v, list) and len(v) == len(ts)})
env_df.insert(0, 'timestamp', ts); env_df.set_index('timestamp', inplace=True)


def _gauge(ax, value, zones, value_label, title, xlim, unit_suffix):
    """Horizontal gauge: colored zones on bottom 60%, marker + value on top 40%."""
    for lo_z, hi_z, color, label in zones:
        ax.axvspan(lo_z, hi_z, color=color, alpha=0.75, ymin=0, ymax=0.6)
        ax.text((lo_z + hi_z) / 2, 0.3, label, ha='center', va='center', fontsize=8)
    if value is not None and value >= 0:
        ax.plot([value], [0.78], marker='v', markersize=16, color='black', zorder=10)
        ax.text(value, 0.93, f'{value}{unit_suffix}', ha='center', fontsize=10, fontweight='bold')
    ax.set_xlim(*xlim); ax.set_ylim(0, 1); ax.set_yticks([])
    ax.set_xlabel(value_label); ax.set_title(title)


# --- Visualization: rich single-snapshot view, or line plot for time series. ---
if len(env_df) == 1:
    row = env_df.iloc[0]

    fig = plt.figure(figsize=(13, 9))
    gs = fig.add_gridspec(3, 2, height_ratios=[1, 1.1, 1.1], hspace=0.55, wspace=0.28)

    # Row 1: gauges
    _gauge(fig.add_subplot(gs[0, 0]),
           value=row.get('heat_index_celsius'),
           zones=[(15, 27, '#b7e4c7', 'Safe'),
                  (27, 32, '#ffd43b', 'Caution'),
                  (32, 41, '#fd7e14', 'Extreme\ncaution'),
                  (41, 54, '#fa5252', 'Danger')],
           value_label='Heat index (°C) — NWS heat-stress scale',
           title=f"Snapshot at {ts[0].strftime('%Y-%m-%d %H:%M')}",
           xlim=(15, 54), unit_suffix=' °C')
    _gauge(fig.add_subplot(gs[0, 1]),
           value=row.get('relative_humidity_percent'),
           zones=[(0, 30, '#f4d35e', 'Dry'),
                  (30, 60, '#b7e4c7', 'Comfortable'),
                  (60, 100, '#74c0fc', 'Humid')],
           value_label='Relative humidity (%)',
           title='Humidity snapshot',
           xlim=(0, 100), unit_suffix=' %')

    # Row 2: thermal-metrics breakdown — shows what amplifies/dampens the raw tile temp.
    ax = fig.add_subplot(gs[1, :])
    thermal = [
        ('Tile temperature',       tile_temp,                                 '#e03131'),
        ('Heat index (feels-like)', row.get('heat_index_celsius'),            '#f76707'),
        ('Apparent temperature',    row.get('apparent_temperature_celsius'),  '#fd7e14'),
        ('Wet-bulb (evap limit)',   row.get('wet_bulb_temperature_celsius'),  '#20c997'),
    ]
    thermal = [(k, v, c) for k, v, c in thermal if v is not None and v >= 0]
    labels, values, colors = zip(*thermal) if thermal else ([], [], [])
    if values:
        bars = ax.barh(labels, values, color=colors)
        for bar, v in zip(bars, values):
            ax.text(v + max(values) * 0.01, bar.get_y() + bar.get_height() / 2,
                    f'{v} °C', va='center', fontsize=10, fontweight='bold')
        ax.set_xlim(0, max(values) * 1.18)
    ax.set_xlabel('°C')
    ax.set_title('Thermal breakdown — tile temp vs perceived heat (gap reveals humidity contribution)')
    ax.invert_yaxis(); ax.grid(axis='x', alpha=0.3)

    # Row 3 left: air-quality contributors (drops −999 sentinels automatically).
    ax = fig.add_subplot(gs[2, 0])
    aq = {
        'PM2.5':     row.get('air_quality_pm2p5:idx'),
        'PM10':      row.get('air_quality_pm10:idx'),
        'O₃':        row.get('air_quality_o3:idx'),
        'SO₂':       row.get('air_quality_so2:idx'),
        'NO₂':       row.get('air_quality_no2:idx'),
        'CO (AQI)':  row.get('aqi_us_co'),
        'Methane':   row.get('methane_ppb'),
    }
    aq = {k: v for k, v in aq.items() if v is not None and v >= 0}
    if aq:
        keys, vals = list(aq.keys()), list(aq.values())
        ax.barh(keys, vals, color='#845ef7')
        for i, v in enumerate(vals):
            ax.text(v + max(vals) * 0.02, i, f'{v}', va='center', fontsize=9)
        ax.set_xlim(0, max(vals) * 1.18)
        ax.set_xlabel('Index value')
        ax.set_title('Air-quality contributors (higher = worse)')
        ax.invert_yaxis(); ax.grid(axis='x', alpha=0.3)
    else:
        ax.text(0.5, 0.5, 'No air-quality data', ha='center', va='center', transform=ax.transAxes)
        ax.axis('off')

    # Row 3 right: solar irradiance — direct sun is the heat source to shade against.
    ax = fig.add_subplot(gs[2, 1])
    solar_bars = {
        'GHI (global)':  solar_cs.get('ghi'),
        'DNI (direct)':  solar_cs.get('dni'),
        'DHI (diffuse)': solar_cs.get('dhi'),
    }
    solar_bars = {k: v for k, v in solar_bars.items() if v is not None and v >= 0}
    if solar_bars:
        keys, vals = list(solar_bars.keys()), list(solar_bars.values())
        ax.barh(keys, vals, color=['#f59f00', '#e67700', '#ffd43b'])
        for i, v in enumerate(vals):
            ax.text(v + max(vals) * 0.02, i, f'{v:.0f}', va='center', fontsize=9)
        ax.set_xlim(0, max(vals) * 1.18)
        ax.set_xlabel('W/m²')
        ax.set_title('Solar irradiance (clear sky) — DNI high ⇒ direct sun dominant, shade highly effective')
        ax.invert_yaxis(); ax.grid(axis='x', alpha=0.3)
    else:
        ax.text(0.5, 0.5, 'No solar data', ha='center', va='center', transform=ax.transAxes)
        ax.axis('off')

    plt.tight_layout(); plt.show()

    # Compact readout + a short driver interpretation.
    hi, app, wb, rh = (row.get('heat_index_celsius'),
                       row.get('apparent_temperature_celsius'),
                       row.get('wet_bulb_temperature_celsius'),
                       row.get('relative_humidity_percent'))
    print(f"\nThermal readout at this hour:")
    print(f"  Heat index           : {hi} °C")
    print(f"  Apparent temperature : {app} °C")
    print(f"  Wet-bulb temperature : {wb} °C")
    print(f"  Relative humidity    : {rh} %")
    if hi is not None and tile_temp is not None:
        gap = tile_temp - hi
        print(f"\nDriver interpretation:")
        if rh is not None and rh < 40:
            print(f"  • Dry air (RH {rh}%) → sweating is effective; evaporative cooling (misting) will work well here.")
        elif rh is not None and rh >= 60:
            print(f"  • Humid air (RH {rh}%) → sweating is suppressed; ventilation/shade beats misting.")
        if wb is not None and wb > 28:
            print(f"  • Wet-bulb {wb} °C approaches human limits — prolonged outdoor waiting is unsafe.")
        print(f"  • Solar load DNI={solar_cs.get('dni', '?')} W/m² is the primary heat input — full shade blocks ~70–90% of this.")
else:
    # Multi-timestamp fallback: line plot through the day.
    plot_cols = [c for c in ('heat_index_celsius', 'apparent_temperature_celsius',
                             'wet_bulb_temperature_celsius', 'relative_humidity_percent')
                 if c in env_df.columns and (env_df[c] >= 0).all()]
    if plot_cols:
        fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
        thermal_cols = [c for c in plot_cols if 'humidity' not in c]
        if thermal_cols:
            env_df[thermal_cols].plot(ax=axes[0], marker='o')
            axes[0].set_title(f"Thermal comfort at ({lat:.4f}, {lon:.4f})")
            axes[0].set_ylabel('°C'); axes[0].grid(alpha=0.3)
        if 'relative_humidity_percent' in env_df.columns:
            env_df['relative_humidity_percent'].plot(ax=axes[1], color='steelblue', marker='o')
            axes[1].set_title('Relative humidity'); axes[1].set_ylabel('%'); axes[1].grid(alpha=0.3)
        plt.tight_layout(); plt.show()

# --- Location on map ---
fmap_env = folium.Map(location=[lat, lon], zoom_start=16, tiles='cartodbpositron')
folium.Marker(
    location=[lat, lon],
    popup=f"Env params<br>({lat:.6f}, {lon:.6f})<br>{tile_temp} °C",
    icon=folium.Icon(color='red', icon='info-sign'),
).add_to(fmap_env)
folium.Circle(
    location=[lat, lon], radius=GRANULARITY_M / 2,
    color='red', fill=True, fill_opacity=0.15,
    popup=f"~{GRANULARITY_M}m tile",
).add_to(fmap_env)

# --- Expose driver values for Step 9 (cached-path compatibility). ---
peak_hi = env_df['heat_index_celsius'].max() if 'heat_index_celsius' in env_df.columns else None
if 'heat_index_celsius' in env_df.columns:
    peak_idx = env_df['heat_index_celsius'].idxmax()
    peak_rh = env_df.loc[peak_idx, 'relative_humidity_percent'] if 'relative_humidity_percent' in env_df.columns else None
    peak_wb = env_df.loc[peak_idx, 'wet_bulb_temperature_celsius'] if 'wet_bulb_temperature_celsius' in env_df.columns else None
else:
    peak_rh = peak_wb = None
print(f"\nExported to Step 9 → peak_hi={peak_hi}°C, peak_rh={peak_rh}%, peak_wb={peak_wb}°C")

fmap_env

---
## Step 9 — Prioritized action list

### What you are doing
Combining everything we learned into a single action-oriented DataFrame. For each of the top stops, a **scoring heuristic** weighs several candidate interventions and returns the top two as `primary_action` and `secondary_action`. The scoring rules:

| Rule | When it fires | What it scores |
|---|---|---|
| **Wet-bulb safety** | `wet_bulb > 30 °C` at peak | Always wins — closes stop until retrofit |
| **Full-coverage canopy** | #1 stop AND `sky_pct > 45%` | Priority scales with temperature delta vs AOI mean |
| **Partial canopy** | `temp delta > 2 °C` AND `veg < 20%` | Priority scales with delta |
| **Street trees** | `vegetation < 15%` | Scales with how far below target + delta |
| **Cool-surface paving** | `impervious > 60%` | Scales with impervious excess + delta |
| **Misting station** | `peak heat index > 30 °C` AND `RH < 40%` | Dry regime — evaporative cooling works |
| **Ventilated shelter** | `peak heat index > 30 °C` AND `RH ≥ 60%` | Humid regime — misting ineffective |
| **Hybrid cooling** | `peak heat index > 30 °C`, moderate RH | Compromise |

### Why this matters
This is the row the council actually reads. Two things make the output defensible:

1. **Humidity picks the intervention type.** The same heat index + dry air calls for misting; the same heat index + humid air calls for ventilation. Past dashboards that ignored humidity frequently recommended the wrong fix.
2. **Stops differentiate on temperature delta, humidity regime, and rank** — not just raw surface percentages. Even when two stops share similar satellite/street-view numbers, their scored primary actions can differ because `delta` and the rank-gated canopy rule shift the ranking.

Every column is traceable:
- `temperature_c` — Step 3 (spatial join)
- `impervious_pct` / `vegetation_pct` — Step 6 (or 6b)
- `primary_action` — derived here, driven by scores combining Steps 3, 6, 7, 8
- `primary_reason` / `secondary_reason` — the exact thresholds that fired, so you can defend each recommendation.

In [ ]:
aoi_mean_t = t_stats.get('mean')
# env-driven peaks — fall through to None if Step 8/8b didn't run.
_peak_rh = globals().get('peak_rh')
_peak_wb = globals().get('peak_wb')


def _recommend(row, *, sky_pct_top1, peak_hi, peak_rh, peak_wb, aoi_mean_t, is_top1):
    """Score candidate interventions, return primary + secondary with rationale."""
    temp  = row.get('temperature_c') or 0
    veg   = row.get('vegetation_pct') if row.get('vegetation_pct') is not None else 0
    imp   = row.get('impervious_pct') if row.get('impervious_pct') is not None else 0
    delta = (temp - aoi_mean_t) if aoi_mean_t is not None else 0

    candidates = []  # (score, action, reason)

    # Wet-bulb safety gate — always wins when triggered.
    if peak_wb is not None and peak_wb > 30:
        candidates.append((20,
            'URGENT: close stop during peak hours until retrofit complete',
            f'wet-bulb {peak_wb} °C — above survivable threshold for prolonged waiting'))

    # Full shade canopy: #1 stop with open sky, OR any stop running hot with low veg.
    if is_top1 and sky_pct_top1 is not None and sky_pct_top1 > 45:
        candidates.append((12 + delta,
            'Install full-coverage shade canopy over shelter',
            f'open-sky {sky_pct_top1}% at hottest stop; tile {temp} °C ({delta:+.1f} vs AOI mean)'))
    elif delta > 2.0 and veg < 20:
        candidates.append((6 + delta,
            'Install partial shade canopy over seating area',
            f'tile {temp} °C ({delta:+.1f} vs mean) with tree cover only {veg}%'))

    # Street trees — mid-term cooling; gated on low vegetation.
    if veg < 15:
        candidates.append((3 + (15 - veg) * 0.3 + delta * 0.4,
            'Plant street trees along pedestrian approach (3–5 yr cooling payoff)',
            f'vegetation {veg}% (target ≥ 15%)'))

    # Reflective / cool-surface paving — gated on high impervious + hot tile.
    if imp > 60:
        candidates.append((4 + (imp - 60) * 0.08 + delta * 0.5,
            'Apply reflective / cool-surface paving around shelter footprint',
            f'impervious {imp}% absorbs solar radiation; tile {delta:+.1f} °C vs mean'))

    # Active cooling — type chosen by humidity regime.
    if peak_hi is not None and peak_hi > 30:
        if peak_rh is not None and peak_rh < 40:
            candidates.append((6 + (peak_hi - 30) + delta * 0.3,
                'Install misting station (evaporative cooling — effective in dry air)',
                f'heat index {peak_hi} °C with RH {peak_rh}% (dry regime)'))
        elif peak_rh is not None and peak_rh >= 60:
            candidates.append((6 + (peak_hi - 30) + delta * 0.3,
                'Retrofit shelter: cross-ventilation + reflective roof (humid regime — misting ineffective)',
                f'heat index {peak_hi} °C with RH {peak_rh}% (humid regime)'))
        else:
            candidates.append((5 + (peak_hi - 30) + delta * 0.3,
                'Hybrid cooling: ventilated shelter + low-volume misting',
                f'heat index {peak_hi} °C with RH {peak_rh}% (moderate humidity)'))

    # Default when nothing triggered.
    if not candidates:
        candidates.append((1,
            'Monitor — below intervention thresholds',
            f'tile {temp} °C; no individual driver exceeds threshold'))

    candidates.sort(key=lambda c: -c[0])
    primary = candidates[0]
    secondary = next((c for c in candidates[1:] if c[1] != primary[1]), None)

    return pd.Series({
        'primary_action':    primary[1],
        'primary_reason':    primary[2],
        'secondary_action':  secondary[1] if secondary else '—',
        'secondary_reason':  secondary[2] if secondary else '—',
    })


actions = top.apply(
    lambda r: _recommend(r,
                         sky_pct_top1=sky_pct if r['rank'] == 1 else None,
                         peak_hi=peak_hi, peak_rh=_peak_rh, peak_wb=_peak_wb,
                         aoi_mean_t=aoi_mean_t,
                         is_top1=r['rank'] == 1),
    axis=1,
)
action_list = pd.concat([top.reset_index(drop=True),
                         actions.reset_index(drop=True)], axis=1)

display_cols = ['rank', 'stop_id', 'name', 'temperature_c',
                'impervious_pct', 'vegetation_pct',
                'primary_action', 'primary_reason',
                'secondary_action', 'secondary_reason']
action_list[display_cols]

In [ ]:
# Export the action list to hand off to the operations team.
out_path = ROOT / 'outputs' / 'bus_stop_action_list.csv'
out_path.parent.mkdir(parents=True, exist_ok=True)
action_list.to_csv(out_path, index=False)
print(f'Saved prioritized action list to {out_path}')

---
## Wrap-up — what you now have

Starting from a single CSV of bus stops you now have:

| Artifact | Step | Audience |
|----------|------|----------|
| Temperature-joined stops table | 3 | GIS / analytics team |
| Visual hotspot ranking on the city map | 4 | Council presentation |
| Above-mean hotspot cluster map | 5 | Council presentation |
| Macro diagnosis of the top-N (impervious / vegetation %) | 6 | Landscape / infrastructure team |
| Street-level confirmation of the #1 stop | 7 | Design review |
| Diurnal heat-index profile at the worst location | 8 | Intervention-type selection |
| Prioritized action list CSV | 9 | Operations hand-off |

Every action in the final list is traceable back to a measurement — not an assumption. That is the defensibility the council was missing, and the reason this workflow scales beyond bus stops.

### Apply this pattern to your other layers

The workflow is agnostic to what kind of asset your points represent. Swap the CSV for any point layer and everything downstream works:

- **Schools / playgrounds** → prioritize which outdoor spaces need tree planting
- **Public benches / transit shelters** → identify which need upgrading to reflective / vented designs
- **Bike-share docks** → identify stations where riders drop off because they overheat
- **Utility substations / pumping stations** → identify infrastructure at heat-failure risk
- **Social-housing units** → identify buildings in hottest blocks for retrofit prioritization

The pattern — **your geometries × our thermal, surface, and environmental layers → ranked actions** — is the whole point.